# Stage 2 — Digital Clock Reader

בשלב זה נאמן מודל לקריאת השעה מתוך תמונת שעון דיגיטלי.

המודל מקבל תמונה ומחזיר שלוש תחזיות:

1. hour — ערכים 0–23
2. minute — ערכים 0–59
3. second — ערכים 0–59

נשתמש ב־ResNet18 pretrained עם שלושה classification heads.

In [42]:
import random
from pathlib import Path

import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms, models
from sklearn.model_selection import train_test_split

In [43]:
BASE_DIR = Path("clock_project")
DATA_DIR = BASE_DIR / "data"
LABELS_PATH = DATA_DIR / "labels.csv"

MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 1
LEARNING_RATE = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)
print("Labels exists:", LABELS_PATH.exists())

Using device: cpu
Labels exists: True


In [44]:
df = pd.read_csv(LABELS_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (20000, 8)


,sample_id,digital_image_path,analog_with_hands_path,analog_clean_path,hour,minute,second,style_name
0,0,clock_project\data\digital\digital_00000.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,23,30,56,classic_light
1,1,clock_project\data\digital\digital_00001.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,17,12,0,minimal_blue
2,2,clock_project\data\digital\digital_00002.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,4,46,58,classic_light
3,3,clock_project\data\digital\digital_00003.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,18,43,26,clean_white
4,4,clock_project\data\digital\digital_00004.png,clock_project\data\analog_with_hands\analog_ha...,clock_project\data\analog_clean\analog_clean_0...,16,39,52,classic_light


In [45]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["style_name"] if "style_name" in df.columns else None
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))

Train size: 16000
Validation size: 4000


In [46]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2,
        hue=0.05
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [47]:
class DigitalClockDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["digital_image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        hour = torch.tensor(int(row["hour"]), dtype=torch.long)
        minute = torch.tensor(int(row["minute"]), dtype=torch.long)
        second = torch.tensor(int(row["second"]), dtype=torch.long)

        return image, hour, minute, second

In [48]:
train_dataset = DigitalClockDataset(train_df, transform=train_transform)
val_dataset = DigitalClockDataset(val_df, transform=val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 500
Validation batches: 125


In [49]:
images, hours, minutes, seconds = next(iter(train_loader))

print("Images:", images.shape)
print("Hours:", hours.shape)
print("Minutes:", minutes.shape)
print("Seconds:", seconds.shape)

print(f"Example label: {hours[0].item():02d}:{minutes[0].item():02d}:{seconds[0].item():02d}")

Images: torch.Size([32, 3, 224, 224])
Hours: torch.Size([32])
Minutes: torch.Size([32])
Seconds: torch.Size([32])
Example label: 01:22:45


In [50]:
class DigitalClockClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()

        self.bottleneck = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.hour_head = nn.Linear(256, 24)
        self.minute_head = nn.Linear(256, 60)
        self.second_head = nn.Linear(256, 60)

    def forward(self, x):
        features = self.backbone(x)
        features = self.bottleneck(features)

        hour_logits = self.hour_head(features)
        minute_logits = self.minute_head(features)
        second_logits = self.second_head(features)

        return hour_logits, minute_logits, second_logits

    def predict_time(self, x):
        hour_logits, minute_logits, second_logits = self.forward(x)

        hour = torch.argmax(hour_logits, dim=1)
        minute = torch.argmax(minute_logits, dim=1)
        second = torch.argmax(second_logits, dim=1)

        return hour, minute, second

In [51]:
model = DigitalClockClassifier().to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

print("Model ready")

Model ready


In [ ]:
history = {
    "train_loss": [],
    "val_loss": [],
    "hour_acc": [],
    "minute_acc": [],
    "second_acc": [],
    "full_time_acc": []
}

best_full_time_acc = 0.0

MODEL_PATH = MODELS_DIR / "digital_time_reader.pth"
HISTORY_PATH = RESULTS_DIR / "digital_reader_training_history.csv"

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for images, hours, minutes, seconds in train_loader:
        images = images.to(DEVICE)
        hours = hours.to(DEVICE)
        minutes = minutes.to(DEVICE)
        seconds = seconds.to(DEVICE)

        optimizer.zero_grad()

        pred_hour, pred_minute, pred_second = model(images)

        loss_hour = criterion(pred_hour, hours)
        loss_minute = criterion(pred_minute, minutes)
        loss_second = criterion(pred_second, seconds)

        # minute and second are harder because they have 60 classes
        loss = loss_hour + 2.0 * loss_minute + 2.0 * loss_second

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0

    correct_hour = 0
    correct_minute = 0
    correct_second = 0
    correct_full = 0
    total = 0

    with torch.no_grad():
        for images, hours, minutes, seconds in val_loader:
            images = images.to(DEVICE)
            hours = hours.to(DEVICE)
            minutes = minutes.to(DEVICE)
            seconds = seconds.to(DEVICE)

            pred_hour, pred_minute, pred_second = model(images)

            loss_hour = criterion(pred_hour, hours)
            loss_minute = criterion(pred_minute, minutes)
            loss_second = criterion(pred_second, seconds)

            loss = loss_hour + 2.0 * loss_minute + 2.0 * loss_second
            val_loss += loss.item()

            hour_preds = torch.argmax(pred_hour, dim=1)
            minute_preds = torch.argmax(pred_minute, dim=1)
            second_preds = torch.argmax(pred_second, dim=1)

            correct_hour += (hour_preds == hours).sum().item()
            correct_minute += (minute_preds == minutes).sum().item()
            correct_second += (second_preds == seconds).sum().item()

            full_correct = (
                (hour_preds == hours) &
                (minute_preds == minutes) &
                (second_preds == seconds)
            )

            correct_full += full_correct.sum().item()
            total += images.size(0)

    val_loss = val_loss / len(val_loader)

    hour_acc = correct_hour / total
    minute_acc = correct_minute / total
    second_acc = correct_second / total
    full_time_acc = correct_full / total

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["hour_acc"].append(hour_acc)
    history["minute_acc"].append(minute_acc)
    history["second_acc"].append(second_acc)
    history["full_time_acc"].append(full_time_acc)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"H: {hour_acc:.3f} | "
        f"M: {minute_acc:.3f} | "
        f"S: {second_acc:.3f} | "
        f"Full: {full_time_acc:.3f}"
    )

    if full_time_acc > best_full_time_acc:
        best_full_time_acc = full_time_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"Saved best model | Full Time Acc: {best_full_time_acc:.3f}")

history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print("Training finished")
print("Best Full Time Accuracy:", best_full_time_acc)
print("Model saved to:", MODEL_PATH)
print("History saved to:", HISTORY_PATH)

In [ ]:
history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["train_loss"], label="Train Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Digital Reader — Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["hour_acc"], label="Hour Accuracy")
plt.plot(history_df["minute_acc"], label="Minute Accuracy")
plt.plot(history_df["second_acc"], label="Second Accuracy")
plt.plot(history_df["full_time_acc"], label="Full Time Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Digital Reader — Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
best_model = DigitalClockClassifier().to(DEVICE)
best_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
best_model.eval()

print("Best model loaded")

In [ ]:
def denormalize_image(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    image = tensor.cpu() * std + mean
    image = torch.clamp(image, 0, 1)

    return image.permute(1, 2, 0).numpy()


best_model.eval()

images, hours, minutes, seconds = next(iter(val_loader))

images = images.to(DEVICE)
hours = hours.to(DEVICE)
minutes = minutes.to(DEVICE)
seconds = seconds.to(DEVICE)

with torch.no_grad():
    pred_hour, pred_minute, pred_second = best_model.predict_time(images)

n = min(6, images.size(0))

plt.figure(figsize=(15, 4))

for i in range(n):
    true_time = f"{hours[i].item():02d}:{minutes[i].item():02d}:{seconds[i].item():02d}"
    pred_time = f"{pred_hour[i].item():02d}:{pred_minute[i].item():02d}:{pred_second[i].item():02d}"

    correct = true_time == pred_time

    plt.subplot(1, n, i + 1)
    plt.imshow(denormalize_image(images[i]))
    plt.title(
        f"T: {true_time}\nP: {pred_time}",
        color="green" if correct else "red",
        fontsize=9
    )
    plt.axis("off")

plt.tight_layout()
plt.show()